Librerias

In [10]:
import pandas as pd
from pathlib import Path

# Sujeto de entrenamiento
SUJETOS_ENTRENAMIENTO = ['SUJETO 1', 'SUJETO 10', 'SUJETO 11', 'SUJETO 12', 'SUJETO 14',
                        'SUJETO 2', 'SUJETO 3', 'SUJETO 4', 'SUJETO 6', 'SUJETO 7']

# Carpetas de frames a considerar (RGB original + augmentations)
FRAME_FOLDERS = ['rgb', 'rgb_FLIP', 'rgb_GRAY', 'rgb_ROTATION', 'rgb_STRETCH']

# Carpeta donde se guardará el inventario
OUTPUT_DIR = Path(r'D:\SALIDA\Landmarks')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Crear carpeta si no existe

def create_dataset_inventory(root_path, sujetos_entrenamiento=SUJETOS_ENTRENAMIENTO):
    root = Path(root_path)
    data = []
    
    for sujeto_dir in sorted(root.iterdir()):
        if not sujeto_dir.is_dir() or sujeto_dir.name not in sujetos_entrenamiento:
            continue
        
        # Usar Path correcto para la carpeta con ñ
        señas_path = sujeto_dir / 'señas_procesadas'
        if not señas_path.exists():
            continue
        
        for clase_dir in sorted(señas_path.iterdir()):
            if not clase_dir.is_dir():
                continue
            
            for muestra_dir in sorted(clase_dir.iterdir()):
                if not muestra_dir.is_dir():
                    continue
                
                for frame_folder in FRAME_FOLDERS:
                    carpeta_frames = muestra_dir / frame_folder
                    if not carpeta_frames.exists():
                        continue
                    
                    # Buscar todos los PNG (mayúsculas o minúsculas)
                    keyframes = sorted(carpeta_frames.glob('*.png')) + sorted(carpeta_frames.glob('*.PNG'))
                    
                    data.append({
                        'sujeto': sujeto_dir.name,
                        'clase': clase_dir.name,
                        'muestra': muestra_dir.name,
                        'tipo': frame_folder,
                        'n_keyframes': len(keyframes),
                        'ruta_muestra': str(carpeta_frames)  # Convertir a string al final
                    })
    
    df = pd.DataFrame(data)
    output_file = OUTPUT_DIR / 'inventario_completo.csv'
    df.to_csv(output_file, index=False, encoding='utf-8')  # Guardar con UTF-8
    
    print(f"Total secuencias registradas: {len(df)}")
    print(f"\nInventario guardado en: {output_file}")
    
    return df

# Ejecutar
root_path = r'D:\SALIDA\SALIDA'
df_inventario = create_dataset_inventory(root_path)


Total secuencias registradas: 3700

Inventario guardado en: D:\SALIDA\Landmarks\inventario_completo.csv


In [ ]:
#Landmarks para val y test



Extraccion de Landmarks

In [11]:
import cv2 
import mediapipe as mp
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Landmarks a extraer
contorno = [109,10,338,297,332,284,251,389,356,454,323,361,288,397,365,379,378,400,377,152,148,176,149,150,136,172,58,132,93,234,127,162,21,54,103,67]
frente = [104,69,108,151,337,299,9]
ceja_der = [70,63,105,66,107]
ceja_izq = [336,296,334,293,300]
ojo_Der_completo = [33,246,161,160,159,158,157,173,133,155,154,153,145,144,163,7,247,30,225,224,29,223,27,222,28,56,221,189,190,224,233,232,231,230,229,228]
ojo_Izq_completo = [362,398,384,385,386,387,388,466,263,249,390,373,374,380,381,382,464,453,452,451,450,449,448,413,414,286,441,258,442,443,257,444,259,260,445,467,342]
nariz = [55,8,285,193,168,417,122,6,351,197,195,5,4,1,19,94,2,97,98,326,327,275,440,344,438,274,134,220,115,131,419,248,281,399,420,196,3,51,45,360,355]
mejilla_der = [139,156,34,143,227,116,123,137,147,177,213,215,192,138,207,214,210,135,187,117,50,205,212,202,216,206]
mejilla_izq = [368,346,383,265,372,264,345,447,426,423,266,330,425,280,352,366,401,433,435,416,367,364,434,432,436,427,411,376]
boca = [62,191,80,81,82,13,312,311,310,415,292,324,318,402,317,14,87,178,88,95]
labios = [57,185,40,39,37,0,267,269,270,409,287,375,321,405,314,17,84,181,91,146]        
barbilla = [169,170,211,204,194,32,140,201,208,171,175,199,200,421,428,396,418,262,369,395,431,424,422]

landmark_indices = (contorno + frente + ceja_der + ceja_izq + 
                   ojo_Der_completo + ojo_Izq_completo + nariz + 
                   mejilla_der + mejilla_izq + boca + labios + barbilla)
landmark_indices_unique = list(dict.fromkeys(landmark_indices))
NUM_LANDMARKS = len(landmark_indices_unique)

print(f"Landmarks a extraer: {NUM_LANDMARKS}")

mp_face_mesh = mp.solutions.face_mesh

def extract_all_landmarks(df_inventario, output_dir=r'D:\SALIDA\Landmarks'):
    output_path = Path(output_dir) / 'npy_landmarks'
    output_path.mkdir(parents=True, exist_ok=True)
    
    errores = []
    procesados = []
    
    with mp_face_mesh.FaceMesh(
        static_image_mode=True,  # procesar cada frame como imagen independiente
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.3
    ) as face_mesh:
        
        for idx, row in tqdm(df_inventario.iterrows(), total=len(df_inventario)):
            try:
                carpeta = Path(row['ruta_muestra'])
                frames_paths = sorted(carpeta.glob('*.png')) + sorted(carpeta.glob('*.PNG'))
                
                if not frames_paths:
                    continue
                
                all_landmarks = []
                
                for frame_path in frames_paths:
                    file_bytes = np.fromfile(str(frame_path), dtype=np.uint8)
                    image = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
                    
                    if image is None:
                        continue
                    
                    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                    results = face_mesh.process(image_rgb)
                    
                    if results.multi_face_landmarks:
                        landmarks = results.multi_face_landmarks[0]
                        coords_selected = np.array([[landmarks.landmark[i].x, 
                                                     landmarks.landmark[i].y, 
                                                     landmarks.landmark[i].z] 
                                                    for i in landmark_indices_unique])
                        all_landmarks.append(coords_selected)
                    else:
                        # NO duplicar frames si no se detecta
                        continue
                
                if not all_landmarks:
                    continue  # ignorar secuencias donde no se detectó ningún frame
                
                landmarks_array = np.array(all_landmarks)
                
                filename = f"{row['sujeto']}_{row['clase']}_{row['muestra']}_{row['tipo']}.npy"
                save_path = output_path / filename
                np.save(save_path, landmarks_array)
                
                procesados.append({
                    'sujeto': row['sujeto'],
                    'clase': row['clase'],
                    'muestra': row['muestra'],
                    'tipo': row['tipo'],
                    'n_keyframes': len(all_landmarks),
                    'landmarks_shape': str(landmarks_array.shape),
                    'landmarks_file': str(save_path)
                })
                
            except Exception as e:
                errores.append({
                    'sujeto': row['sujeto'],
                    'clase': row['clase'],
                    'muestra': row['muestra'],
                    'tipo': row['tipo'],
                    'error': str(e)
                })
    
    df_procesados = pd.DataFrame(procesados)
    df_procesados.to_csv(output_path / 'landmarks_procesados.csv', index=False, encoding='utf-8')
    
    if errores:
        df_errores = pd.DataFrame(errores)
        df_errores.to_csv(output_path / 'landmarks_errores.csv', index=False, encoding='utf-8')
    
    print(f"\nProcesados: {len(procesados)}")
    print(f"Errores: {len(errores)}")
    
    return df_procesados

# Ejecutar
df = pd.read_csv(r'D:\SALIDA\Landmarks\inventario_completo.csv', encoding='utf-8')
df_landmarks = extract_all_landmarks(df)


Landmarks a extraer: 283


100%|██████████| 3700/3700 [1:07:49<00:00,  1.10s/it]


Procesados: 3700
Errores: 0
